In [2]:
from langchain.agents import create_agent

In [19]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model
summarization_model=init_chat_model("gemini-2.5-flash",model_provider="google_genai")
agentcreate=init_chat_model("openai/gpt-oss-20b",model_provider="groq")
agent=create_agent(
    model=agentcreate,
    
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model=summarization_model,
        trigger=("messages",10),
        keep=("messages",4)
    )])


In [20]:

config={"configurable":{"thread_id":"test-1"}}

In [21]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='39cfbccd-a66e-4bbf-94a4-d18e84719a0b'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks "What is 2+2?" It\'s a simple math question. The answer is 4. No further context. Just give answer.'}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 78, 'total_tokens': 129, 'completion_time': 0.10027688, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.005182706, 'prompt_tokens_details': None, 'queue_time': 0.390371853, 'total_time': 0.105459586}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_f102a3179c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a18c8b-f079-75d0-8092-a79c35acfc8e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 51, 'total_tokens': 1

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


In [29]:
def search_hotels(city:str):
     return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""
agent=create_agent(
     model=init_chat_model("openai/gpt-oss-20b",model_provider="groq"),
     checkpointer=InMemorySaver(),
     middleware=[SummarizationMiddleware(
         model=init_chat_model("gemini-2.5-flash",model_provider="google_genai"),
         trigger=("tokens",1200),
         keep=("tokens",500)
     )],
)

In [30]:
config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [31]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, Messages: {len(response['messages'])}")

Paris: ~1005 tokens, Messages: 2
London: ~1323 tokens, Messages: 3
Tokyo: ~658 tokens, Messages: 3
New York: ~1857 tokens, Messages: 5
Dubai: ~1494 tokens, Messages: 3
Singapore: ~1745 tokens, Messages: 3


In [33]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider="groq"),
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=init_chat_model("gemini-2.5-flash", model_provider="google_genai"),
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response["messages"][-1].content)  # Print the last message content

Paris: ~166 tokens (0.1297%), 4 msgs
Here are a few options in Paris, ranging from luxury to budget-friendly:

| Hotel | Price (per night) | Highlights |
|-------|-------------------|------------|
| **Grand Hotel** | $350 | 5‑star, central location, spa, Michelin‑star dining |
| **City Inn** | $180 | Boutique 4‑star, art‑inspired rooms, rooftop terrace |
| **Budget Stay** | $75 | 3‑star, clean & comfortable, close to public transport |

**What would you like next?**  
- Want to see more hotels or filter by price range?  
- Interested in booking a room?  
- Need details on amenities, parking, or nearby attractions?
London: ~259 tokens (0.2023%), 8 msgs
It looks like the search returned the same set of hotels as for Paris—probably a hiccup on our end. Could you let me know if you’d still like a list of hotels in London (or if there’s a specific type or price range you’re interested in)? I’ll pull the correct information for you right away.
Tokyo: ~499 tokens (0.3898%), 12 msgs
It looks l

Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).
Compliance workflows where human oversight is mandatory.
Long-running conversations where human feedback guides the agent.

In [82]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool


In [84]:
@tool
def read_email(email:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email}"

@tool
def send_email(recipient: str, subject: str, body: str)->str:
    """Mock function to send an email."""
    return  f"Email sent to {recipient} with subject '{subject}'"

In [85]:
agent=create_agent(
    model=init_chat_model("openai/gpt-oss-20b",model_provider="groq"),
    checkpointer=InMemorySaver(),
    tools=[read_email,send_email],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email":False

        }
    )])

In [86]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4268f68b-c879-4b87-9f10-a55813a10066'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email function. Provide JSON with body, recipient, subject.', 'tool_calls': [{'id': 'fc_c4e05018-740d-4a25-a2c6-a6d20b272651', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 171, 'total_tokens': 226, 'completion_time': 0.061531606, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.009106804, 'prompt_tokens_details': None, 'queue_time': 0.485604737, 'total_time': 0.07063841}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e2cb7a84ec', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'l

In [67]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Email sent to john@test.com with subject **"Hello"** and body **"How are you?"**. Let me know if you need anything else!


In [68]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0d3d2fcc-37a5-48e0-8bc1-43b089647c3d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function send_email.', 'tool_calls': [{'id': 'fc_32bdee5a-3d1a-45be-805d-27e1f29ba297', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 171, 'total_tokens': 216, 'completion_time': 0.048452986, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008438846, 'prompt_tokens_details': None, 'queue_time': 0.212064654, 'total_time': 0.056891832}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='l

In [72]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m sorry, but I can’t send the email right now. If you’d like me to try again or need help with something else, just let me know!


In [87]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email",
                            "args": {
                                "recipient": "john@example.com",
                                "subject": "Updated Subject",
                                "body": "Updated Body",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✅ Result: Your email has been sent to **john@example.com** with the subject **“Updated Subject”** and body **“Updated Body.”** Let me know if you’d like to send another message or make any changes!


In [ ]:
print(send_email.name)
print(send_email.args)

AttributeError: 'function' object has no attribute 'name'